# 05. Heatmap과 Pairplot

## 학습 목표
- Heatmap으로 상관계수 매트릭스 시각화
- Pairplot으로 전체 변수 관계 한눈에 파악
- Iris 데이터로 품종 특성 구분
- 다변량 패턴 발견 및 변수 선택

---

## 1. 왜 다변량 분석이 필요한가?

**제조업 실무 사례:**
- 다변수 품질 요인 분석 (온도, 압력, 속도 등)
- 센서 데이터 간 상관관계 탐지
- 중복 변수 제거로 모델 단순화

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# Iris 데이터 로드
iris = sns.load_dataset('iris')
print(f"전체 샘플: {len(iris)}개")
print(f"품종: {iris['species'].unique()}")
iris.head()

## 2. 상관계수 매트릭스 계산

**비즈니스 질문:** 어떤 측정치들이 서로 연관되어 있나?

In [ ]:
# 수치형 변수만 선택
numeric_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
corr_matrix = iris[numeric_cols].corr()

print("상관계수 매트릭스:")
print(corr_matrix.round(3))

# 💡 인사이트: petal_length와 petal_width 강한 양의 상관(0.963) → 중복 측정 가능성

## 3. Heatmap 기본: 상관계수 시각화

**비즈니스 질문:** 한눈에 변수 간 관계를 파악할 수 있나?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Heatmap 생성
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={'shrink': 0.8},
            ax=ax)

ax.set_title('Iris 변수 간 상관계수 Heatmap', fontsize=14, fontweight='bold', pad=15)

# 레이블 한글화
labels = ['꽃받침 길이', '꽃받침 너비', '꽃잎 길이', '꽃잎 너비']
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels, rotation=0)

plt.tight_layout()
plt.show()

# 💡 인사이트: 꽃잎 측정치(길이/너비) 간 r=0.963 → 하나만 측정해도 충분할 가능성
# 💡 꽃받침 너비는 다른 변수들과 약한 상관 → 독립적 정보 제공

## 4. Heatmap 고급: 마스크와 삼각형

**개선점:** 대칭 매트릭스는 절반만 표시

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# 상삼각 마스크 생성
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Heatmap with mask
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', 
            cmap='RdYlGn', center=0, square=True, linewidths=2,
            cbar_kws={'shrink': 0.8, 'label': '상관계수'},
            ax=ax, vmin=-1, vmax=1)

ax.set_title('Iris 상관계수 (하삼각 행렬)', fontsize=14, fontweight='bold', pad=15)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels, rotation=0)

plt.tight_layout()
plt.show()

# 💡 인사이트: 불필요한 중복 제거로 가독성 향상
# 💡 RdYlGn 컬러맵으로 양/음의 상관관계 직관적 구분

## 5. Pairplot 기본: 모든 조합 시각화

**비즈니스 질문:** 품종을 구분하는 최적의 변수 조합은?

In [ ]:
# Pairplot 생성
g = sns.pairplot(iris, hue='species', palette='Set1', 
                 height=2.5, aspect=1, 
                 diag_kind='kde',  # 대각선은 KDE
                 plot_kws={'alpha': 0.6, 's': 50, 'edgecolor': 'black', 'linewidth': 0.5})

g.fig.suptitle('Iris 전체 변수 Pairplot (품종별)', fontsize=14, fontweight='bold', y=1.01)

plt.tight_layout()
plt.show()

# 💡 인사이트: petal_length vs petal_width 조합에서 세 품종이 가장 명확히 구분됨
# 💡 setosa는 모든 변수에서 다른 품종과 분리 → 식별 용이
# 💡 versicolor와 virginica는 꽃잎 측정치에서만 구분 가능

## 6. Pairplot 고급: 특정 변수만 선택

**효율화:** 핵심 변수 2-3개만 집중 분석

In [ ]:
# 꽃잎 관련 변수만
g = sns.pairplot(iris, vars=['petal_length', 'petal_width', 'sepal_length'],
                 hue='species', palette='husl',
                 height=3, aspect=1.2,
                 diag_kind='hist',
                 plot_kws={'alpha': 0.7, 's': 60},
                 diag_kws={'edgecolor': 'black', 'linewidth': 1.5})

g.fig.suptitle('Iris 핵심 변수 Pairplot', fontsize=14, fontweight='bold', y=1.01)

plt.tight_layout()
plt.show()

# 💡 인사이트: petal_length 단독으로도 setosa 완벽 구분 가능 (< 2.5cm)
# 💡 petal_width 추가 시 virginica와 versicolor 구분 정확도 향상

## 7. 실전 분석: 자동차 데이터 Heatmap

**제조업 적용: mpg 데이터로 연비 영향 요인 분석**

In [ ]:
# MPG 데이터 로드
mpg = sns.load_dataset('mpg')

# 수치형 변수만 선택
mpg_numeric = mpg[['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']].dropna()
mpg_corr = mpg_numeric.corr()

fig, ax = plt.subplots(figsize=(10, 8))

# Heatmap 생성
sns.heatmap(mpg_corr, annot=True, fmt='.2f', cmap='Spectral',
            center=0, square=True, linewidths=2, linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': '상관계수'},
            ax=ax, vmin=-1, vmax=1)

ax.set_title('자동차 속성 상관관계 Heatmap', fontsize=14, fontweight='bold', pad=15)

# 레이블 한글화
mpg_labels = ['연비', '실린더', '배기량', '마력', '무게', '가속']
ax.set_xticklabels(mpg_labels, rotation=45, ha='right')
ax.set_yticklabels(mpg_labels, rotation=0)

plt.tight_layout()
plt.show()

# 강한 상관관계 추출
print("\n연비(mpg)와의 상관계수:")
print(mpg_corr['mpg'].sort_values(ascending=False))

# 💡 인사이트: 무게(-0.83), 배기량(-0.81), 실린더(-0.78) 모두 강한 음의 상관
# 💡 이 세 변수는 서로 0.9 이상 상관 → 다중공선성 문제
# 💡 가속은 연비와 약한 양의 상관(0.42) → 독립적 정보

## 8. Clustermap: 계층적 군집화

**고급 기법:** 유사한 변수끼리 자동 그룹화

In [ ]:
# Clustermap 생성
g = sns.clustermap(mpg_corr, annot=True, fmt='.2f', cmap='vlag',
                   center=0, linewidths=1, linecolor='gray',
                   cbar_kws={'label': '상관계수'},
                   figsize=(10, 9), vmin=-1, vmax=1)

g.fig.suptitle('자동차 속성 Clustermap (계층적 군집)', fontsize=14, fontweight='bold', y=0.98)

plt.show()

# 💡 인사이트: [cylinders, displacement, weight, horsepower] 하나의 클러스터 형성
# 💡 이들은 '차량 크기/성능' 요인을 나타내는 중복 변수
# 💡 회귀 분석 시 이 중 하나만 선택하는 것이 바람직

## 9. 실무 의사결정: 변수 선택 전략

**비즈니스 질문:** 최소한의 변수로 최대 정보를 얻으려면?

In [ ]:
# VIF(Variance Inflation Factor) 개념 시각화
high_corr_pairs = []
threshold = 0.8

for i in range(len(mpg_corr.columns)):
    for j in range(i+1, len(mpg_corr.columns)):
        if abs(mpg_corr.iloc[i, j]) > threshold:
            high_corr_pairs.append({
                '변수1': mpg_corr.columns[i],
                '변수2': mpg_corr.columns[j],
                '상관계수': mpg_corr.iloc[i, j]
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('상관계수', key=abs, ascending=False)

print("\n다중공선성 위험 변수 쌍 (|r| > 0.8):")
print(high_corr_df)

# 시각화
fig, ax = plt.subplots(figsize=(10, 6))

y_pos = np.arange(len(high_corr_df))
labels = [f"{row['변수1']} - {row['변수2']}" for _, row in high_corr_df.iterrows()]

bars = ax.barh(y_pos, high_corr_df['상관계수'].abs(), 
               color=plt.cm.Reds(high_corr_df['상관계수'].abs()),
               edgecolor='black', linewidth=1.2)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel('상관계수 (절댓값)', fontsize=12)
ax.set_title('다중공선성 위험 변수 쌍', fontsize=14, fontweight='bold')
ax.axvline(0.8, color='red', linestyle='--', linewidth=2, label='임계값 (0.8)')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# 💡 인사이트: displacement-weight (0.93), cylinders-displacement (0.95) 특히 주의
# 💡 권장 변수 세트: weight (대표), acceleration (독립), model_year (시간)
# 💡 cylinders, displacement, horsepower는 weight로 대체 가능

---

## 📊 핵심 요약

### 코드 패턴 (30%)
```python
# 상관계수 계산
corr_matrix = df[numeric_cols].corr()

# Heatmap
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0)

# Pairplot
sns.pairplot(df, hue='category', diag_kind='kde')

# Clustermap
sns.clustermap(corr_matrix, cmap='vlag', center=0)
```

### 도출된 인사이트 (70%)
1. **변수 중복**: Iris의 petal_length-petal_width (r=0.96) → 하나만 사용 가능
2. **품종 구분**: setosa는 모든 변수에서 분리, versicolor-virginica는 꽃잎 측정치 필요
3. **자동차 분석**: weight, displacement, cylinders가 같은 정보 제공 → weight만 선택
4. **독립 변수**: acceleration은 다른 변수와 낮은 상관 → 추가 정보 제공

### 실무 의사결정
- **변수 선택**: |r| > 0.8인 변수 쌍에서 하나만 선택
- **모델 개발**: 다중공선성 제거로 해석력 향상
- **측정 비용**: 중복 센서 제거로 비용 절감

### Heatmap vs Pairplot 선택
| 목적 | 도구 | 장점 |
|------|------|------|
| 빠른 전체 파악 | Heatmap | 수치 정확, 공간 효율 |
| 패턴 탐색 | Pairplot | 비선형 관계, 이상치 발견 |
| 변수 그룹화 | Clustermap | 자동 군집, 계층 구조 |

### 주의사항
- Heatmap은 선형 관계만 표현 → 비선형은 Pairplot으로 확인
- 변수 많을 때 Pairplot은 느림 → 핵심 변수만 선택
- 상관계수 0에 가까워도 비선형 관계 존재 가능